# Index

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from index.inverted_index import Index
from index.documents import load_dataset, tokenizer


# index = Index().build_index(rcv1, tokenizer)

In [3]:
# rcv1 = load_dataset("rcv1")

In [4]:
# i = 0
# x = rcv1.data[i]
# y = rcv1.target[i]
# sid = rcv1.sample_id[i]

# print("sid", sid)
# print("x", x)
# print("y", y)

In [5]:
# from sklearn.cluster import MiniBatchKMeans

# X = rcv1.data # CSR matrix (804414 x 47236)
# mbk = MiniBatchKMeans(n_clusters=20, batch_size=1000, random_state=0)
# mbk.fit(X)                          # supports sparse input

# # cluster centers are dense (20 x n_features)
# centers = mbk.cluster_centers_
# # top features per cluster
# import numpy as np
# for i, c in enumerate(centers):
#     top_idxs = np.argsort(-c)[:10]
#     print("cluster", i, "top feature indices:", top_idxs)

In [6]:
msmarco = load_dataset("msmarco-passage/dev/judged")

In [7]:
print(msmarco)

Dataset(id='msmarco-passage/dev/judged', provides=['docs', 'queries', 'qrels'])


In [8]:
import locale, sys
print(sys.flags.utf8_mode)                 # should be 1
print(locale.getpreferredencoding(False))  # should be UTF-8

1
utf-8


In [9]:
msmarco_index = Index()
msmarco_index.build_index(msmarco, tokenizer, max_num_docs=10001)

{'the': [(1, 6),
  (2, 1),
  (3, 5),
  (4, 8),
  (5, 4),
  (6, 4),
  (7, 5),
  (8, 14),
  (9, 4),
  (10, 6),
  (12, 2),
  (13, 2),
  (14, 4),
  (15, 4),
  (16, 6),
  (17, 3),
  (18, 5),
  (19, 3),
  (20, 8),
  (21, 6),
  (22, 2),
  (23, 2),
  (24, 3),
  (25, 3),
  (27, 5),
  (28, 6),
  (29, 1),
  (30, 5),
  (31, 3),
  (32, 5),
  (33, 5),
  (34, 1),
  (35, 8),
  (36, 6),
  (37, 8),
  (38, 7),
  (39, 2),
  (41, 1),
  (42, 1),
  (43, 2),
  (45, 5),
  (46, 5),
  (47, 4),
  (48, 5),
  (49, 1),
  (50, 2),
  (51, 1),
  (52, 6),
  (53, 1),
  (54, 4),
  (55, 1),
  (56, 2),
  (57, 5),
  (58, 1),
  (59, 4),
  (60, 1),
  (61, 1),
  (62, 5),
  (63, 17),
  (64, 6),
  (65, 3),
  (66, 3),
  (67, 1),
  (68, 5),
  (69, 6),
  (70, 9),
  (71, 5),
  (72, 2),
  (73, 8),
  (74, 7),
  (75, 1),
  (76, 3),
  (77, 8),
  (78, 6),
  (79, 5),
  (80, 2),
  (81, 4),
  (82, 3),
  (83, 7),
  (84, 8),
  (85, 8),
  (86, 10),
  (87, 6),
  (88, 3),
  (89, 2),
  (91, 2),
  (93, 1),
  (94, 1),
  (97, 4),
  (99, 1),
  (100, 3

In [10]:
from scipy.sparse import csr_matrix
import numpy as np

def calculate_tf_idf(index, doc_id):
    tf_idf_vector = {}
    for term in index.get_terms():
        tf = index.get_term_frequency(term, doc_id)
        idf = index.get_inverse_document_frequency(term)
        tf_idf_vector[term] = tf * idf
    return tf_idf_vector

def create_tf_idf_matrix(index):
    tf_idf_matrix = {}
    for doc_id in index.get_doc_ids():
        tf_idf_matrix[doc_id] = calculate_tf_idf(index, doc_id)
    return tf_idf_matrix

def create_sparse_tf_idf_matrix(index, dtype=np.float32):
    terms = list(index.get_terms())
    term_to_col = {t:i for i,t in enumerate(terms)}
    doc_ids = index.get_doc_ids()
    doc_to_row = {did:i for i,did in enumerate(doc_ids)}

    rows, cols, data = [], [], []
    for term, posting in index.index.items():
        idf = index.get_inverse_document_frequency(term)
        col = term_to_col[term]
        for doc_id, tf in posting:
            rows.append(doc_to_row[doc_id])
            cols.append(col)
            data.append(np.log10(1+tf) * idf)

    data = np.array(data, dtype=dtype)
    mat = csr_matrix((data, (rows, cols)), shape=(len(doc_ids), len(terms)), dtype=dtype)
    return mat, doc_ids, terms

In [11]:
# tf_idf_matrix = create_tf_idf_matrix(msmarco_index)
# print("TF-IDF vector for document 1:", tf_idf_matrix[1])
# print("tf idf values", list(tf_idf_matrix.values().__iter__())[:10])

mat, doc_ids, terms = create_sparse_tf_idf_matrix(msmarco_index)
print("Sparse TF-IDF matrix shape:", mat.shape)
print("Document IDs:", doc_ids[:10])
print("Terms:", terms[:10])

# bytearray(b'\x00\x01\x02')

Sparse TF-IDF matrix shape: (10001, 37678)
Document IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Terms: ['the', 'presence', 'of', 'communication', 'amid', 'scientific', 'minds', 'was', 'equally', 'important']


In [12]:
import pandas as pd

# df = pd.DataFrame(tf_idf_matrix).T
# print("TF-IDF DataFrame shape:", df.shape)

df = pd.DataFrame.sparse.from_spmatrix(mat, index=doc_ids, columns=terms)
print("DF shape:", df.shape)

DF shape: (10001, 37678)


In [13]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

norm_df = normalize(df, norm="l2") #! cosa fa , è fondamentele se non la metti clusteing fa schifo 

kmeans = KMeans(n_clusters=10, random_state=0) #! use jaccard distance instead of euclidean

clusters = kmeans.fit(norm_df)
print("Cluster centers:", kmeans.cluster_centers_)

document_clusters = kmeans.labels_
print("Document clusters:", document_clusters)

Cluster centers: [[0.00752568 0.         0.00602444 ... 0.         0.         0.        ]
 [0.0067194  0.00033002 0.00923948 ... 0.         0.         0.        ]
 [0.00532604 0.         0.00957934 ... 0.         0.         0.        ]
 ...
 [0.00609501 0.00028023 0.00824213 ... 0.         0.         0.        ]
 [0.00622354 0.         0.01312373 ... 0.         0.         0.        ]
 [0.00608792 0.00184435 0.01099571 ... 0.         0.         0.        ]]
Document clusters: [1 5 5 ... 5 4 4]


In [14]:
medoids = []

for i in range(kmeans.n_clusters):
    cluster_docs = np.flatnonzero(document_clusters == i)
    cluster_vectors = norm_df[cluster_docs]
    medoid_idx = np.argmin(np.linalg.norm(cluster_vectors - kmeans.cluster_centers_[i], axis=1))
    medoids.append(cluster_docs[medoid_idx])

# medoid_vectors = norm_df[medoids].toarray()

medoid_vectors = norm_df[medoids]
nonzero_cols = medoid_vectors.getnnz(axis=0) > 0
medoid_vectors = medoid_vectors[:, nonzero_cols].toarray()

print("Medoids:", medoids)
print("Medoid vectors shape:", medoid_vectors.shape)
print("Medoid vectors:", medoid_vectors)

# for i in range(10):
#     cluster_docs = [doc_id for doc_id, cluster in enumerate(document_clusters) if cluster == i]
#     new_docs_ordering[i] = cluster_docs
#     print(f"Cluster {i} has {len(cluster_docs)} documents.")

# from itertools import islice

# # medoids contiene row index della matrice (0-based)
# target_rows = set(medoids)
# medoid_docs = {}

# for row_idx, doc in enumerate(islice(msmarco.docs_iter(), len(doc_ids))):
#     if row_idx in target_rows:
#         medoid_docs[row_idx] = doc
#     if len(medoid_docs) == len(target_rows):
#         break

# # esempio: testo del primo medoid
# print(medoid_docs[medoids[0]].text)

# # opzionale: stampa tutti i medoid
# for m in medoids:
#     print(f"\nMedoid row {m}")
#     print(medoid_docs[m].text[:500])

Medoids: [np.int64(8089), np.int64(1287), np.int64(9084), np.int64(5298), np.int64(5087), np.int64(7328), np.int64(9937), np.int64(1915), np.int64(259), np.int64(8398)]
Medoid vectors shape: (10, 360)
Medoid vectors: [[0.0112747  0.01455121 0.         ... 0.         0.21803866 0.22683487]
 [0.00433475 0.00990914 0.         ... 0.         0.         0.        ]
 [0.01146728 0.         0.         ... 0.         0.         0.        ]
 ...
 [0.00348141 0.         0.         ... 0.         0.         0.        ]
 [0.00359941 0.         0.         ... 0.         0.         0.        ]
 [0.00453563 0.02073674 0.         ... 0.         0.         0.        ]]


In [15]:
from sklearn.metrics import pairwise_distances
import numpy as np

def calculate_distance_matrix(norm_df, medoid_vectors, method="euclidean"):
    # medoid_vectors = norm_df[medoids]  # CSR (k, d)

    if method == "euclidean":
        return pairwise_distances(medoid_vectors, metric="euclidean")
    elif method == "cosine":
        return pairwise_distances(medoid_vectors, metric="cosine")
    elif method == "jaccard":
        # Jaccard in sklearn works on boolean/binary features
        return pairwise_distances((medoid_vectors > 0).astype(np.uint8), metric="jaccard")
    else:
        raise ValueError(f"Unknown method: {method}")

In [16]:
# medoid_vectors = norm_df[medoids]
# print("Medoid vectors shape:", medoid_vectors.shape)
# print("Medoid vectors:", medoid_vectors)
# medoid_vectors.toarray()  
# print("Medoid vectors:", medoid_vectors)

dist_matrix = calculate_distance_matrix(norm_df, medoid_vectors, method="jaccard")
print("Distance matrix shape:", dist_matrix.shape)
print("Distance matrix:\n", dist_matrix)

Distance matrix shape: (10, 10)
Distance matrix:
 [[0.         0.94059406 0.85483871 0.93258427 0.93406593 0.92708333
  0.95       0.87640449 0.97333333 0.91304348]
 [0.94059406 0.         0.99029126 0.95081967 0.93442623 0.9375
  0.95652174 0.92741935 0.98148148 0.96153846]
 [0.85483871 0.99029126 0.         0.94252874 0.95555556 0.92473118
  0.98305085 0.8452381  0.97222222 0.875     ]
 [0.93258427 0.95081967 0.94252874 0.         0.92727273 0.85185185
  0.90909091 0.9        0.95744681 0.88372093]
 [0.93406593 0.93442623 0.95555556 0.92727273 0.         0.90434783
  0.95121951 0.88181818 0.96907216 0.8988764 ]
 [0.92708333 0.9375     0.92473118 0.85185185 0.90434783 0.
  0.93023256 0.87826087 0.96078431 0.85714286]
 [0.95       0.95652174 0.98305085 0.90909091 0.95121951 0.93023256
  0.         0.96511628 0.96875    0.93333333]
 [0.87640449 0.92741935 0.8452381  0.9        0.88181818 0.87826087
  0.96511628 0.         0.98019802 0.85227273]
 [0.97333333 0.98148148 0.97222222 0.95744

c:\Python311\Lib\site-packages\sklearn\metrics\pairwise.py:2361: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


In [17]:
import networkx as nx
tsp = nx.approximation.traveling_salesman_problem

tsp_graph = nx.from_numpy_array(dist_matrix)
tsp_path = tsp(tsp_graph, cycle=False)
print("TSP path (medoid indices):", tsp_path)


TSP path (medoid indices): [2, 0, 8, 3, 6, 5, 9, 7, 4, 1]


In [18]:
def reorder_documents_by_tsp(document_clusters, medoids, tsp_path):
    """
    Reorder documents by cluster order induced by TSP over medoids.
    Inside each cluster, keep documents in their original order (row order).

    Args:
        document_clusters: array-like of shape (n_docs,), cluster label per doc.
        medoids: list of medoid doc indices (not required for ordering here, kept for API compatibility).
        tsp_path: list of cluster ids in the order returned by TSP.

    Returns:
        ordered_doc_ids: list of document row indices in final order.
        cluster_to_docs: dict {cluster_id: [doc_ids in original order]}.
    """
    document_clusters = np.asarray(document_clusters)

    # Build cluster -> docs preserving natural/original order
    cluster_to_docs = {
        c: np.flatnonzero(document_clusters == c).tolist()
        for c in range(len(medoids))
    }

    # Concatenate clusters following TSP order
    ordered_doc_ids = []
    for c in tsp_path:
        ordered_doc_ids.extend(cluster_to_docs.get(c, []))

    return ordered_doc_ids, cluster_to_docs

In [19]:
ordered_doc_ids, cluster_docs = reorder_documents_by_tsp(document_clusters, medoids, tsp_path)
print("First 30 docs in reordered list:", ordered_doc_ids[:30])

First 30 docs in reordered list: [114, 128, 130, 131, 138, 155, 176, 191, 200, 274, 280, 283, 288, 421, 439, 443, 459, 461, 540, 592, 593, 595, 598, 605, 657, 700, 704, 706, 729, 733]


In [ ]:
from itertools import islice

docs = list(islice(msmarco.docs_iter(), len(document_clusters)))

class OrderedDataset:
    def __init__(self, docs, order):
        self.docs = docs
        self.order = order
    def docs_iter(self):
        for i in self.order:
            yield self.docs[i]

ordered_dataset = OrderedDataset(docs, ordered_doc_ids)
reordered_inverted_index = Index()
reordered_inverted_index.build_index(
    ordered_dataset,
    tokenizer,
    max_num_docs=-1,
    d_gap=True
)

stat_r = reordered_inverted_index.get_approx_size()

msmarco_index = Index()
msmarco_index.build_index(msmarco, tokenizer, max_num_docs=100, d_gap=True)

stat_m = msmarco_index.get_approx_size()
print(f"Original index size: {stat_m} bytes")
print(f"Reordered index size: {stat_r} bytes")

ind_1 = reordered_inverted_index.index
ind_2 = msmarco_index.index

print("Msmarco index size:", len(msmarco_index.index))


                === Index Size Breakdown ===
                Total Size: 1,146,815 bytes (1,119.94 KB / 1.09 MB)
                - Term Bytes: 0 bytes (0.00%)
                - Doc ID Bytes: 765,998 bytes (748.04 KB) (66.79%)
                - TF Bytes: 380,817 bytes (371.89 KB) (33.21%)
                

                === Index Size Breakdown ===
                Total Size: 1,137,694 bytes (1,111.03 KB / 1.08 MB)
                - Term Bytes: 0 bytes (0.00%)
                - Doc ID Bytes: 756,904 bytes (739.16 KB) (66.53%)
                - TF Bytes: 380,790 bytes (371.87 KB) (33.47%)
                
Original index size: 1137694 bytes
Reordered index size: 1146815 bytes
Msmarco index size: 37675


In [21]:
msmarco_index.remove_tf()
msmarco_index.compress_index("gamma")
msmarco_index.save_index("msmarco_index.bin")

In [ ]:
test_index = Index()
test_index.clear_index()
test_index.load_index("msmarco_index.bin")

In [ ]:
print("Test index loaded. Number of terms:", len(test_index.index))

In [ ]:
#test compressions 
import copy

gamma_index_r = copy.deepcopy(reordered_inverted_index)
delta_index_r = copy.deepcopy(reordered_inverted_index)
vb_index_r = copy.deepcopy(reordered_inverted_index)

gamma_index_r.compress_index("gamma")
delta_index_r.compress_index("delta")
vb_index_r.compress_index("varbyte")

gamma_index = copy.deepcopy(msmarco_index)
delta_index = copy.deepcopy(msmarco_index)
vb_index = copy.deepcopy(msmarco_index)

gamma_index.compress_index("gamma")
delta_index.compress_index("delta")
vb_index.compress_index("varbyte")

print("\n" + "="*70)
print(f"{'ORIGINAL (NOT REORDERED) INDEX':^70}")
print("="*70)
print("Uncompressed:")
msmarco_index.get_approx_size()

print("\nGamma Compressed:")
gamma_index.remove_tf()
gamma_index.get_approx_size()


print("\nDelta Compressed:")
delta_index.remove_tf()
delta_index.get_approx_size()

print("\nVarbyte Compressed:")
vb_index.remove_tf()
vb_index.get_approx_size()

print("\n" + "="*70)
print(f"{'REORDERED INDEX (TSP)':^70}")
print("="*70)
print("Uncompressed:")
reordered_inverted_index.get_approx_size()

print("\nGamma Compressed:")
gamma_index_r.remove_tf()
gamma_index_r.get_approx_size()

print("\nDelta Compressed:")
delta_index_r.remove_tf()
delta_index_r.get_approx_size()

print("\nVarbyte Compressed:")
vb_index_r.remove_tf()
vb_index_r.get_approx_size()

In [ ]:
from index.inverted_index import Index

test = Index()
num = 4
# res = test._Index__varbyte_encode(num)
# print(f"Varbyte encoding of {num}: {res}")
# for value in test._Index__varbyte_decode(res):
#     print(value)

# byte_val = int(res, 2)
# print(f"Decoded value: {byte_val}")

res_2 = test._Index__gamma_encode(num)
print(f"Gamma encoding of {num}: {res_2}")
test._Index__gamma_decode(res_2)

# res_3 = test._Index__delta_encode(num)
# print(f"Delta encoding of {num}: {res_3}")
# test._Index__delta_decode(res_3)

# d_gaps_list = [3,4,7,21,9,12,2,17]
# res_4_list, outliers, min_ = test.pfor_delta_encode(d_gaps_list) 
# print(f"PForDelta encoding of {d_gaps_list}: {res_4_list}")
# print(f"Outliers: {outliers}")
# print(f"Minimum value: {min_}")
# #! how do i distinguish outliers from non-outliers in the decode? assuming that its encoded as contiguous block of bits

# print(test.pfor_delta_decode(res_4_list, outliers, min_))